# 07.2 — EDA dos Recém-Nascidos

**Purpose:** Descriptive EDA, clinical examples, temporal analysis.
Loads shared artifacts produced by `07.1_carga_rn.py`.

## 1. Setup

In [ ]:
from __future__ import annotations
import sys, warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

warnings.filterwarnings("ignore")

ROOT    = Path(__file__).resolve().parent.parent
ART_DIR = ROOT / "out" / "artifacts_rn"
OUT_DIR = ROOT / "out" / "07.2_eda_rn"
PLOT_DIR = OUT_DIR / "plots"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

print(f"ROOT    : {ROOT}")
print(f"ART_DIR : {ART_DIR}")

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma

_T = {
    "pt": {
        "peso_title":          "Peso ao Nascer (g)",
        "apgar_title":         "Apgar 5 min",
        "cid_neo_title":       "Top CIDs Neonatais",
        "obito_peso_xlabel":   "% obito",
        "obito_peso_title":    "Obito por Faixa de Peso",
        "obito_cid_xlabel":    "% obito neonatal",
        "obito_cid_title":     "Obito por CID Materno",
        "peso_ig_title":       "Peso Medio por IG Materna",
        "perfil_suptitle":     "Perfil dos Recem-Nascidos",
        "apgar_scatter_xlabel":"Apgar 1min",
        "apgar_scatter_ylabel":"Apgar 5min",
        "apgar_scatter_title": "Apgar 1min vs 5min\n(azul=vivo, vermelho=obito)",
        "apgar_cat_title":     "Apgar 5min por Categoria de Peso",
        "apgar_cat_xlabel":    "Apgar 5min",
        "apgar_dep_xlabel":    "% Apgar deprimido (<7)",
        "apgar_dep_title":     "Apgar Deprimido por CID Materno",
        "apgar_suptitle":      "Analise de Apgar",
        "neo_obito_xlabel":    "Taxa de Obito Neonatal (%)",
        "neo_obito_title":     "Obito Neonatal por Perfil Historico Materno (G/A/N)",
        "heatmap_title":       "Taxa de Obito Neonatal (%) - IG Materna x Peso ao Nascer",
        "heatmap_xlabel":      "Peso ao Nascer",
        "heatmap_ylabel":      "IG Materna",
        "tend_xlabel":         "Ano de Nascimento",
        "tend_ylabel":         "Taxa (%)",
        "tend_title":          "Tendencias Temporais — RN",
        "mae_obito_xlabel":    "Faixa Etária Materna",
        "mae_obito_ylabel":    "% Óbito Neonatal",
        "mae_obito_title":     "Óbito Neonatal por Faixa Etária Materna",
        "mae_peso_ylabel":     "% Baixo Peso",
        "mae_peso_title":      "Baixo Peso RN por Faixa Etária Materna",
        "mae_pmed_ylabel":     "Peso Mediano (g)",
        "mae_pmed_title":      "Peso Mediano RN por Faixa Etária Materna",
        "mae_suptitle":        "Biometria dos RN por Faixa Etária Materna\n(bins 5 em 5 anos — captura gravidez precoce/abuso)",
        "lbl_2500":            "2.500g",
        "lbl_dep":             "<7 deprimido",
        "lbl_apgar5":          "Apgar 5min=7",
        "lbl_apgar1":          "Apgar 1min=7",
        "lbl_mortalidade":     "Mortalidade neonatal",
        "lbl_baixo_peso":      "Baixo peso",
        "lbl_prematuro":       "Prematuro",
        "lbl_apgar_dep":       "Apgar dep.",
    },
    "en": {
        "peso_title":          "Birth Weight (g)",
        "apgar_title":         "Apgar 5 min",
        "cid_neo_title":       "Top Neonatal ICD-10 Codes",
        "obito_peso_xlabel":   "% death",
        "obito_peso_title":    "Death by Weight Category",
        "obito_cid_xlabel":    "% neonatal death",
        "obito_cid_title":     "Death by Maternal ICD-10",
        "peso_ig_title":       "Mean Birth Weight by Maternal GA",
        "perfil_suptitle":     "Neonatal Profile",
        "apgar_scatter_xlabel":"Apgar 1min",
        "apgar_scatter_ylabel":"Apgar 5min",
        "apgar_scatter_title": "Apgar 1min vs 5min\n(blue=alive, red=death)",
        "apgar_cat_title":     "Apgar 5min by Weight Category",
        "apgar_cat_xlabel":    "Apgar 5min",
        "apgar_dep_xlabel":    "% depressed Apgar (<7)",
        "apgar_dep_title":     "Depressed Apgar by Maternal ICD-10",
        "apgar_suptitle":      "Apgar Analysis",
        "neo_obito_xlabel":    "Neonatal Death Rate (%)",
        "neo_obito_title":     "Neonatal Death by Maternal Obstetric Profile (G/A/N)",
        "heatmap_title":       "Neonatal Death Rate (%) — Maternal GA × Birth Weight",
        "heatmap_xlabel":      "Birth Weight",
        "heatmap_ylabel":      "Maternal GA",
        "tend_xlabel":         "Birth Year",
        "tend_ylabel":         "Rate (%)",
        "tend_title":          "Temporal Trends — Neonates",
        "mae_obito_xlabel":    "Maternal Age Group",
        "mae_obito_ylabel":    "% Neonatal Death",
        "mae_obito_title":     "Neonatal Death by Maternal Age Group",
        "mae_peso_ylabel":     "% Low Birth Weight",
        "mae_peso_title":      "Low Birth Weight by Maternal Age Group",
        "mae_pmed_ylabel":     "Median Weight (g)",
        "mae_pmed_title":      "Median Birth Weight by Maternal Age Group",
        "mae_suptitle":        "Neonatal Biometrics by Maternal Age Group\n(5-year bins)",
        "lbl_2500":            "2,500g",
        "lbl_dep":             "<7 depressed",
        "lbl_apgar5":          "Apgar 5min=7",
        "lbl_apgar1":          "Apgar 1min=7",
        "lbl_mortalidade":     "Neonatal mortality",
        "lbl_baixo_peso":      "Low birth weight",
        "lbl_prematuro":       "Preterm",
        "lbl_apgar_dep":       "Depressed Apgar",
    },
}
T = _T[LANG]

## Load artifacts

In [ ]:
with open(ART_DIR / "rn_dataset.pkl", "rb") as f:
    _art = pickle.load(f)

df          = _art["df"]
df_rn_atend = _art["df_rn_atend"]
print(f"df           : {len(df):,} bebes")
print(f"df_rn_atend  : {len(df_rn_atend):,} atendimentos RN")

## Section 1 — Perfil Geral (6 paineis)

In [ ]:
fig, axes = plt.subplots(
    2, 3,
    figsize=(20, 11),
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1.05, 1.05, 1.55]}
)
axes = axes.flatten()

# Peso
df["peso_g"].dropna().hist(ax=axes[0], bins=40, color="steelblue", edgecolor="white")
for v, c, l in [(2500, "red", T["lbl_2500"]), (1500, "orange", "1.500g"), (4000, "green", "4.000g")]:
    axes[0].axvline(v, color=c, ls="--", lw=1.5, label=l)
axes[0].set_title(T["peso_title"])
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Apgar 5min
df["apgar_5min"].dropna().hist(ax=axes[1], bins=10, color="seagreen", edgecolor="white")
axes[1].axvline(7, color="red", ls="--", lw=1.5, label=T["lbl_dep"])
axes[1].set_title(T["apgar_title"])
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

# Top CIDs neonatais
CID_MAP = {
    "P073": "Prematuridade extrema", "P221": "Angustia resp.",
    "P229": "Desconforto resp.",     "P228": "Taquipneia transit.",
    "P589": "Ictericia",             "O809": "Parto normal",
    "Z039": "Obs. s/diagnostico",    "P220": "SRDS tipo I"
}
_cid_col = "cid_rn" if "cid_rn" in df.columns else None
if _cid_col:
    tc = df[_cid_col].value_counts().dropna().head(7)
    tc.index = [CID_MAP.get(c, c) for c in tc.index]
    axes[2].barh(tc.index[::-1], tc.values[::-1], color="steelblue")
    axes[2].set_title(T["cid_neo_title"])
    axes[2].grid(axis="x", alpha=0.3)
else:
    axes[2].text(0.5, 0.5, "CID indisponivel", ha="center", va="center",
                 transform=axes[2].transAxes)

# Obito por faixa de peso
fp = (
    df.groupby("faixa_peso", observed=True)["obito_rn"]
    .agg(["mean", "count"])
    .reset_index()
)
fp["mean"] *= 100
cors = ["#d62728" if v > 20 else "#ff7f0e" if v > 5 else "#2ca02c" for v in fp["mean"]]
axes[3].barh(fp["faixa_peso"].astype(str)[::-1], fp["mean"][::-1], color=cors[::-1])
axes[3].set_xlabel(T["obito_peso_xlabel"])
axes[3].set_title(T["obito_peso_title"])
axes[3].grid(axis="x", alpha=0.3)
for i, (_, row) in enumerate(fp[::-1].iterrows()):
    axes[3].text(row["mean"] + 0.2, i, f"n={int(row['count'])}", va="center", fontsize=8)

# Obito por segmento materno
if "segmento_mae" in df.columns:
    seg = (
        df.groupby("segmento_mae")["obito_rn"]
        .agg(["mean", "count"])
        .reset_index()
        .query("count >= 10")
        .sort_values("mean", ascending=False)
    )
    seg["mean"] *= 100
    cs2 = ["#d62728" if v > 2 else "#ff7f0e" if v > 0.5 else "#2ca02c" for v in seg["mean"]]
    axes[4].barh(seg["segmento_mae"][::-1], seg["mean"][::-1], color=cs2[::-1])
    axes[4].set_xlabel(T["obito_cid_xlabel"])
    axes[4].set_title(T["obito_cid_title"])
    axes[4].grid(axis="x", alpha=0.3)
else:
    axes[4].text(0.5, 0.5, "segmento_mae indisponivel", ha="center", va="center",
                 transform=axes[4].transAxes)

# Peso por IG materna
if df["ig_mae"].notna().sum() > 50:
    df["ig_grp"] = pd.cut(
        df["ig_mae"],
        bins=[0, 28, 32, 34, 37, 40, 50],
        labels=["<28s", "28-31s", "32-33s", "34-36s", "37-40s", ">40s"],
        right=False
    )
    ig_p = df.groupby("ig_grp", observed=True)["peso_g"].mean().dropna()
    axes[5].bar(ig_p.index.astype(str), ig_p.values, color="steelblue")
    axes[5].axhline(2500, color="red", ls="--", lw=1, label=T["lbl_2500"])
    axes[5].set_title(T["peso_ig_title"])
    axes[5].legend(fontsize=8)
    axes[5].grid(axis="y", alpha=0.3)
    plt.setp(axes[5].xaxis.get_majorticklabels(), rotation=30)
else:
    axes[5].text(0.5, 0.5, "IG insuficiente", ha="center", va="center",
                 transform=axes[5].transAxes, fontsize=12)

plt.suptitle(T["perfil_suptitle"], fontsize=14, y=1.01)
plt.savefig(PLOT_DIR / "01_perfil_geral_rn.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("[S1] 01_perfil_geral_rn.png salvo.")

## Section 2 — Analise de Apgar (NEW)

In [ ]:
print("\n" + "="*60)
print("SECTION 2 — Analise de Apgar")
print("="*60)

fig2, axes2 = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

# Plot 1: Apgar 1min vs 5min scatter colored by obito_rn
_a = df[df["apgar_1min"].notna() & df["apgar_5min"].notna()].copy()
_colors = _a["obito_rn"].map({0: "steelblue", 1: "#d62728"})
axes2[0].scatter(
    _a["apgar_1min"], _a["apgar_5min"],
    c=_colors, alpha=0.4, s=15, rasterized=True
)
axes2[0].axhline(7, color="red", ls="--", lw=1, label=T["lbl_apgar5"])
axes2[0].axvline(7, color="orange", ls="--", lw=1, label=T["lbl_apgar1"])
axes2[0].set_xlabel(T["apgar_scatter_xlabel"])
axes2[0].set_ylabel(T["apgar_scatter_ylabel"])
axes2[0].set_title(T["apgar_scatter_title"])
axes2[0].legend(fontsize=8)
axes2[0].grid(alpha=0.3)

# Plot 2: Apgar 5min by weight category
_bins  = [0, 1500, 2500, 4000, 10000]
_lbls  = ["<1.5kg", "1.5-2.5kg", "2.5-4kg", ">4kg"]
_df_a2 = df[df["apgar_5min"].notna() & df["peso_g"].notna()].copy()
_df_a2["cat_peso_ap"] = pd.cut(_df_a2["peso_g"], bins=_bins, labels=_lbls, right=False)
_df_a2.groupby("cat_peso_ap", observed=True)["apgar_5min"].plot.kde(
    ax=axes2[1], legend=True
)
axes2[1].axvline(7, color="red", ls="--", lw=1)
axes2[1].set_title(T["apgar_cat_title"])
axes2[1].set_xlabel(T["apgar_cat_xlabel"])
axes2[1].grid(alpha=0.3)

# Plot 3: Apgar deprimido rate by maternal CID segment
if "segmento_mae" in df.columns and df["apgar_dep"].notna().sum() > 50:
    _ap3 = (
        df.groupby("segmento_mae")["apgar_dep"]
        .agg(["mean", "count"])
        .reset_index()
        .query("count >= 10")
        .sort_values("mean", ascending=False)
    )
    _ap3["mean"] *= 100
    _c3 = ["#d62728" if v > 10 else "#ff7f0e" if v > 5 else "#2ca02c" for v in _ap3["mean"]]
    axes2[2].barh(_ap3["segmento_mae"][::-1], _ap3["mean"][::-1], color=_c3[::-1])
    axes2[2].set_xlabel(T["apgar_dep_xlabel"])
    axes2[2].set_title(T["apgar_dep_title"])
    axes2[2].grid(axis="x", alpha=0.3)
else:
    axes2[2].text(0.5, 0.5, "Dados insuficientes", ha="center", va="center",
                  transform=axes2[2].transAxes)

plt.suptitle(T["apgar_suptitle"], fontsize=13)
plt.savefig(PLOT_DIR / "02_analise_apgar.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("[S2] 02_analise_apgar.png salvo.")

# Apgar por grupo CSV
if "segmento_mae" in df.columns:
    apgar_grupo = (
        df.groupby("segmento_mae")["apgar_dep"]
        .agg(["mean", "count", "sum"])
        .rename(columns={"mean": "taxa_apgar_dep", "count": "n", "sum": "n_apgar_dep"})
        .reset_index()
        .sort_values("taxa_apgar_dep", ascending=False)
    )
    apgar_grupo["taxa_apgar_dep"] = (apgar_grupo["taxa_apgar_dep"] * 100).round(2)
    apgar_grupo.to_csv(OUT_DIR / "apgar_por_grupo.csv", index=False)
    print("[S2] apgar_por_grupo.csv salvo.")

## Section 3 — Exemplo 1: Perfil G/A/N x Obito

In [ ]:
print("\n" + "="*60)
print("EXEMPLO 1 — Perfil Historico Materno x Obito Neonatal")
print("="*60)

if "perfil_historico" in df.columns and df["perfil_historico"].notna().sum() > 50:
    perfil_ob = (
        df.groupby("perfil_historico")["obito_rn"]
        .agg(["count", "sum", "mean"])
        .rename(columns={"count": "n", "sum": "obitos", "mean": "taxa"})
        .reset_index()
        .query("n >= 10")
        .sort_values("taxa", ascending=False)
    )
    perfil_ob["taxa_pct"] = (perfil_ob["taxa"] * 100).round(2)

    ref = perfil_ob[perfil_ob["perfil_historico"] == "PRIMIGESTA"]["taxa"].values
    if len(ref) and ref[0] > 0:
        perfil_ob["RR_primigesta"] = (perfil_ob["taxa"] / ref[0]).round(2)
    else:
        perfil_ob["RR_primigesta"] = np.nan

    print(perfil_ob[["perfil_historico", "n", "obitos", "taxa_pct", "RR_primigesta"]].to_string(index=False))

    fig3, ax3 = plt.subplots(figsize=(9, 5))
    cores_p = ["#d62728" if v > 1 else "#ff7f0e" if v > 0.3 else "#2ca02c"
               for v in perfil_ob["taxa_pct"]]
    ax3.barh(perfil_ob["perfil_historico"][::-1], perfil_ob["taxa_pct"][::-1], color=cores_p[::-1])
    for i, (_, row) in enumerate(perfil_ob[::-1].reset_index(drop=True).iterrows()):
        rr = f"RR={row['RR_primigesta']:.1f}x" if not pd.isna(row["RR_primigesta"]) else ""
        ax3.text(row["taxa_pct"] + 0.02, i, f"n={int(row['n'])} {rr}", va="center", fontsize=8)
    ax3.set_xlabel(T["neo_obito_xlabel"])
    ax3.set_title(T["neo_obito_title"])
    ax3.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "03_obito_por_perfil_materno.png", dpi=150)
    plt.show()
    plt.close()
    print("[S3] 03_obito_por_perfil_materno.png salvo.")
else:
    print("  Perfil historico insuficiente — expanda extracao LLM/regex.")

## Section 4 — Exemplo 2: Mae hipertensiva x Desfecho RN (with chi2 table)

In [ ]:
print("\n" + "="*60)
print("EXEMPLO 2 — Mae Hipertensiva (O10-O16) x Desfecho RN")
print("="*60)

_mask_pe  = df["cid_mae"].fillna("").str.match(r"^O1[0-6]")
_mask_npe = ~_mask_pe & df["cid_mae"].notna()

comparacao_rows = []
for nome, mask in [("Com PE/HAS (O10-O16)", _mask_pe), ("Sem PE/HAS", _mask_npe)]:
    sub = df[mask]
    if len(sub) == 0:
        continue
    print(f"\n  {nome} (n={len(sub):,}):")
    print(f"    Obito neonatal : {sub['obito_rn'].sum():,} ({sub['obito_rn'].mean()*100:.2f}%)")
    print(f"    Baixo peso     : {sub['baixo_peso'].sum():,.0f} ({sub['baixo_peso'].mean()*100:.1f}%)")
    if sub["peso_g"].notna().sum() > 5:
        print(f"    Peso medio     : {sub['peso_g'].mean():.0f}g")
    if sub["apgar_5min"].notna().sum() > 5:
        print(f"    Apgar 5min     : {sub['apgar_5min'].mean():.1f}  deprimido: {sub['apgar_dep'].sum():,.0f}")

    row = {"grupo": nome, "n": len(sub)}
    for col in ["obito_rn", "baixo_peso", "apgar_dep", "prematuro"]:
        if col in sub.columns:
            _s = pd.to_numeric(sub[col], errors="coerce").fillna(0)
            row[f"{col}_pct"] = round(_s.mean() * 100, 2)
    comparacao_rows.append(row)

# Chi2 p-values — tabela 2x2 via numpy (evita problemas de índice do pd.crosstab)
print("\n  Tabela de comparacao com chi2:")
sub_pe  = df[_mask_pe]
sub_npe = df[_mask_npe]
for col, lbl in [("obito_rn", "Obito"), ("baixo_peso", "Baixo peso"),
                  ("apgar_dep", "Apgar dep."), ("prematuro", "Prematuro")]:
    if col not in df.columns:
        continue
    _pe_arr  = pd.to_numeric(sub_pe[col],  errors="coerce").fillna(0).values.astype(int)
    _npe_arr = pd.to_numeric(sub_npe[col], errors="coerce").fillna(0).values.astype(int)
    n_pe_pos  = int(_pe_arr.sum());  n_pe_neg  = len(_pe_arr)  - n_pe_pos
    n_npe_pos = int(_npe_arr.sum()); n_npe_neg = len(_npe_arr) - n_npe_pos
    ct = np.array([[n_pe_pos, n_pe_neg], [n_npe_pos, n_npe_neg]])
    try:
        _, pval, _, _ = chi2_contingency(ct)
    except Exception:
        pval = np.nan
    _pe_pct  = round(_pe_arr.mean()  * 100, 2)
    _npe_pct = round(_npe_arr.mean() * 100, 2)
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    print(f"    {lbl:15s}: PE/HAS={_pe_pct}%  noPE={_npe_pct}%  p={pval:.4f} {sig}")

if comparacao_rows:
    pd.DataFrame(comparacao_rows).to_csv(OUT_DIR / "comparacao_pe_has.csv", index=False)
    print("[S4] comparacao_pe_has.csv salvo.")

## Section 5 — Exemplo 3: Heatmap IG x Peso x Obito

In [ ]:
print("\n" + "="*60)
print("EXEMPLO 3 — Heatmap: IG Materna x Peso ao Nascer x Obito")
print("="*60)

df_hm = df[df["ig_mae"].notna() & df["peso_g"].notna()].copy()

if len(df_hm) > 50:
    df_hm["cat_ig"]   = pd.cut(df_hm["ig_mae"],
                                bins=[0, 32, 34, 37, 40, 50],
                                labels=["<32s", "32-33s", "34-36s", "37-40s", ">40s"],
                                right=False)
    df_hm["cat_peso"] = pd.cut(df_hm["peso_g"],
                                bins=[0, 1500, 2500, 4000, 10000],
                                labels=["<1.5kg", "1.5-2.5kg", "2.5-4kg", ">4kg"],
                                right=False)

    pivot = (
        df_hm.groupby(["cat_ig", "cat_peso"], observed=True)["obito_rn"]
        .agg(["mean", "count"])
        .unstack("cat_peso")["mean"] * 100
    )

    if not pivot.empty:
        fig5, ax5 = plt.subplots(figsize=(9, 5))
        sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
                    linewidths=0.5, ax=ax5, annot_kws={"size": 10})
        ax5.set_title(T["heatmap_title"])
        ax5.set_xlabel(T["heatmap_xlabel"])
        ax5.set_ylabel(T["heatmap_ylabel"])
        plt.tight_layout()
        plt.savefig(PLOT_DIR / "04_heatmap_ig_peso.png", dpi=150)
        plt.show()
        plt.close()
        print("[S5] 04_heatmap_ig_peso.png salvo.")
    else:
        print("  Dados insuficientes para heatmap.")
else:
    print("  Amostra insuficiente para o heatmap.")

## Section 6 — Analise Temporal (NEW)

In [ ]:
print("\n" + "="*60)
print("SECTION 6 — Analise Temporal dos RN")
print("="*60)

if "ano_nascimento" in df.columns and df["ano_nascimento"].notna().sum() > 50:
    _temp = (
        df.groupby("ano_nascimento")
        .agg(
            total_nascimentos=("obito_rn", "count"),
            obitos=("obito_rn", "sum"),
            n_baixo_peso=("baixo_peso", "sum"),
            n_prematuro=("prematuro", "sum"),
            n_apgar_dep=("apgar_dep", "sum"),
        )
        .reset_index()
    )
    _temp["taxa_obito"]     = (_temp["obitos"]      / _temp["total_nascimentos"] * 100).round(2)
    _temp["taxa_baixo_peso"] = (_temp["n_baixo_peso"] / _temp["total_nascimentos"] * 100).round(2)
    _temp["taxa_prematuro"] = (_temp["n_prematuro"]  / _temp["total_nascimentos"] * 100).round(2)
    _temp["taxa_apgar_dep"] = (_temp["n_apgar_dep"]  / _temp["total_nascimentos"] * 100).round(2)
    _temp = _temp[_temp["total_nascimentos"] >= 20].sort_values("ano_nascimento")

    print(_temp[["ano_nascimento", "total_nascimentos", "taxa_obito",
                  "taxa_baixo_peso", "taxa_prematuro", "taxa_apgar_dep"]].to_string(index=False))

    _temp.to_csv(OUT_DIR / "temporal_rn_por_ano.csv", index=False)
    print("[S6] temporal_rn_por_ano.csv salvo.")

    fig6, ax6 = plt.subplots(figsize=(11, 5))
    _anos = _temp["ano_nascimento"].astype(str)
    ax6.plot(_anos, _temp["taxa_obito"],      "o-", color="#d62728", label=f"{T['lbl_mortalidade']} (%)")
    ax6.plot(_anos, _temp["taxa_baixo_peso"], "s-", color="#ff7f0e", label=f"{T['lbl_baixo_peso']} (%)")
    ax6.plot(_anos, _temp["taxa_prematuro"],  "^-", color="#1f77b4", label=f"{T['lbl_prematuro']} (%)")
    ax6.plot(_anos, _temp["taxa_apgar_dep"],  "D-", color="#9467bd", label=f"{T['lbl_apgar_dep']} (%)")
    ax6.set_xlabel(T["tend_xlabel"])
    ax6.set_ylabel(T["tend_ylabel"])
    ax6.set_title(T["tend_title"])
    ax6.legend(fontsize=9)
    ax6.grid(alpha=0.3)
    plt.setp(ax6.xaxis.get_majorticklabels(), rotation=30)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "05_temporal_rn.png", dpi=150)
    plt.show()
    plt.close()
    print("[S6] 05_temporal_rn.png salvo.")
else:
    print("  Dados de ano insuficientes — verifique coluna ano_nascimento.")

## Section 7 — Biometria por CID Materno (NEW)

In [ ]:
print("\n" + "="*60)
print("SECTION 7 — Biometria por CID Materno")
print("="*60)

if "segmento_mae" in df.columns:
    _top6 = df["segmento_mae"].value_counts().head(6).index.tolist()
    _bio  = df[df["segmento_mae"].isin(_top6)].copy()

    bio_rows = []
    for seg, grp in _bio.groupby("segmento_mae"):
        _n = len(grp)
        _w = grp["peso_g"].dropna()
        _a = grp["apgar_5min"].dropna()
        bio_rows.append({
            "segmento_mae": seg,
            "n": _n,
            "peso_mediana_g": round(_w.median(), 1) if len(_w) else np.nan,
            "peso_q1":        round(_w.quantile(0.25), 1) if len(_w) else np.nan,
            "peso_q3":        round(_w.quantile(0.75), 1) if len(_w) else np.nan,
            "apgar5_mediana": round(_a.median(), 1) if len(_a) else np.nan,
            "apgar5_q1":      round(_a.quantile(0.25), 1) if len(_a) else np.nan,
            "apgar5_q3":      round(_a.quantile(0.75), 1) if len(_a) else np.nan,
            "pct_prematuro":  round(pd.to_numeric(grp["prematuro"], errors="coerce").mean() * 100, 2),
            "pct_baixo_peso": round(pd.to_numeric(grp["baixo_peso"], errors="coerce").mean() * 100, 2),
        })

    df_bio = pd.DataFrame(bio_rows).sort_values("n", ascending=False)
    df_bio["peso_IQR"]   = df_bio.apply(
        lambda r: f"{r['peso_mediana_g']} [{r['peso_q1']}-{r['peso_q3']}]", axis=1
    )
    df_bio["apgar5_IQR"] = df_bio.apply(
        lambda r: f"{r['apgar5_mediana']} [{r['apgar5_q1']}-{r['apgar5_q3']}]", axis=1
    )

    print(df_bio[["segmento_mae", "n", "peso_IQR", "apgar5_IQR",
                   "pct_prematuro", "pct_baixo_peso"]].to_string(index=False))
    df_bio.to_csv(OUT_DIR / "biometria_por_cid_materno.csv", index=False)
    print("[S7] biometria_por_cid_materno.csv salvo.")

# Biometria por faixa etária materna (bins 5 em 5, início 10a)
if "idade_mae" in df.columns and pd.to_numeric(df["idade_mae"], errors="coerce").notna().sum() > 50:
    df["faixa_etaria_mae"] = pd.cut(
        pd.to_numeric(df["idade_mae"], errors="coerce"),
        bins=[0, 10, 15, 20, 25, 30, 35, 40, 45, 50, 120],
        labels=["<10", "10-14", "15-19", "20-24", "25-29",
                "30-34", "35-39", "40-44", "45-49", "≥50"],
        right=False
    )
    bio_mae = (
        df.groupby("faixa_etaria_mae", observed=True)
        .agg(
            n=("obito_rn", "count"),
            pct_obito=("obito_rn", "mean"),
            pct_baixo_peso=("baixo_peso_flag", "mean"),
            pct_prematuro=("prematuro", "mean"),
            peso_mediana=("peso_g", "median"),
        )
        .reset_index()
    )
    for c in ["pct_obito", "pct_baixo_peso", "pct_prematuro"]:
        bio_mae[c] = (bio_mae[c] * 100).round(2)
    bio_mae["peso_mediana"] = bio_mae["peso_mediana"].round(1)
    print("\nBiometria RN por Faixa Etária Materna:")
    print(bio_mae.to_string(index=False))
    bio_mae.to_csv(OUT_DIR / "biometria_por_faixa_etaria_mae.csv", index=False)

    fig_mae, axes_mae = plt.subplots(1, 3, figsize=(16, 5))
    tab_m = bio_mae.dropna(subset=["pct_obito"])

    cores_ob = ["#d62728" if v > 2 else "#ff7f0e" if v > 0.5 else "#2ca02c"
                for v in tab_m["pct_obito"]]
    axes_mae[0].bar(tab_m["faixa_etaria_mae"].astype(str), tab_m["pct_obito"],
                    color=cores_ob, edgecolor="white", alpha=0.85)
    axes_mae[0].set_xlabel(T["mae_obito_xlabel"]); axes_mae[0].set_ylabel(T["mae_obito_ylabel"])
    axes_mae[0].set_title(T["mae_obito_title"])
    axes_mae[0].tick_params(axis="x", rotation=35); axes_mae[0].grid(axis="y", alpha=0.3)

    axes_mae[1].bar(tab_m["faixa_etaria_mae"].astype(str), tab_m["pct_baixo_peso"],
                    color="steelblue", edgecolor="white", alpha=0.85)
    axes_mae[1].set_xlabel(T["mae_obito_xlabel"]); axes_mae[1].set_ylabel(T["mae_peso_ylabel"])
    axes_mae[1].set_title(T["mae_peso_title"])
    axes_mae[1].tick_params(axis="x", rotation=35); axes_mae[1].grid(axis="y", alpha=0.3)

    axes_mae[2].bar(tab_m["faixa_etaria_mae"].astype(str), tab_m["peso_mediana"],
                    color="seagreen", edgecolor="white", alpha=0.85)
    axes_mae[2].axhline(2500, color="red", ls="--", lw=1, label=T["lbl_2500"])
    axes_mae[2].set_xlabel(T["mae_obito_xlabel"]); axes_mae[2].set_ylabel(T["mae_pmed_ylabel"])
    axes_mae[2].set_title(T["mae_pmed_title"])
    axes_mae[2].legend(fontsize=8); axes_mae[2].tick_params(axis="x", rotation=35)
    axes_mae[2].grid(axis="y", alpha=0.3)

    plt.suptitle(T["mae_suptitle"], fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "07_biometria_faixa_etaria_mae.png", dpi=150, bbox_inches="tight")
    plt.show(); plt.close()
    print("[S7] 07_biometria_faixa_etaria_mae.png salvo.")
else:
    print("  segmento_mae indisponivel.")

In [ ]:
print("\n" + "="*60)
print("  07.2 EDA RN — CONCLUIDO")
print("="*60)
print(f"  Plots  : {PLOT_DIR}")
print(f"  CSVs   : {OUT_DIR}")
print("  Arquivos:")
for _f in sorted(OUT_DIR.glob("*.csv")):
    print(f"    {_f.name}")
for _f in sorted(PLOT_DIR.glob("*.png")):
    print(f"    plots/{_f.name}")
print("="*60)